In [ ]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [ ]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'f_phase2_train'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
CFG['resume'] = os.environ.get('NCA_RESUME', '0') == '1'
CFG['allow_device_change'] = os.environ.get('NCA_ALLOW_DEVICE_CHANGE', '0') == '1'
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [ ]:
# functions passed through c_models/output; definitions only, no training side effects
runtime_nb = nbformat.read(INPUT / 'runtime.ipynb', as_version=4)
for runtime_cell in runtime_nb.cells:
    exec(compile(runtime_cell.source, 'runtime.ipynb', 'exec'), globals())
PACK = torch.load(INPUT / 'dataset.pt', weights_only=True, map_location='cpu')
SPECS = CFG['models']

In [ ]:
# content identities for exact checkpoint resumption
data_hash = hashlib.sha256((INPUT / 'dataset.pt').read_bytes()).hexdigest()
code_hash = (INPUT / 'runtime.sha256').read_text()
run_rows = []

In [ ]:
# Phase 1 initialization is specific to the same model / regime / replicate
phase1_manifest = pd.read_csv(INPUT / 'phase1_manifest.csv')

In [ ]:
# selected array row or the whole serial stage; no shared manifest writes from workers
jobs = pd.read_csv(ROOT / 'job_matrix.csv')
selected_index = os.environ.get('NCA_RUN_INDEX', '')
if selected_index:
    jobs = jobs.loc[jobs['index'].eq(int(selected_index))]
runs = OUTPUT / 'phase2_runs'
runs.mkdir(exist_ok=True)
for job in jobs.to_dict('records'):
    spec = next(item for item in SPECS if item['name'] == job['model'])
    name, regime, replicate = job['model'], job['regime'], job['replicate']
    destination = runs / job['run']
    warm_start = None
    previous = phase1_manifest.query('model == @name and regime == @regime and replicate == @replicate').iloc[0]
    if CFG['phase2_initialization'] == 'phase1':
        warm_start = INPUT / 'phase1_runs' / previous['checkpoint']
    external = os.environ.get('NCA_WARM_START', '')
    if external:
        warm_start = Path(external)
    result = fit_run(spec, CFG, PACK, 2, regime, replicate, destination, data_hash, code_hash, warm_start=warm_start)
    atomic_json(result, destination / 'result.json')